# Laboratory 2 homework — Rijden de Treinen analytics

DuckDB is used for analytics; PostgreSQL stores the `disruptions` table (OLTP-style).
Large files use the `.big` suffix (ignored by git).


## 1. Download data


In [1]:
import urllib.request
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

UA = {"User-Agent": "Mozilla/5.0 (compatible; AGH-MLOps-homework/1.0)"}

DOWNLOADS = [
    (
        "stations",
        "https://opendata.rijdendetreinen.nl/public/stations/stations-2023-09.csv",
        DATA / "stations-2023-09.big.csv",
    ),
    (
        "distances",
        "https://opendata.rijdendetreinen.nl/public/tariff-distances/tariff-distances-2022-01.csv",
        DATA / "tariff-distances-2022-01.big.csv",
    ),
]
for year in range(2011, 2024):
    DOWNLOADS.append(
        (
            f"disruptions-{year}",
            f"https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-{year}.csv",
            DATA / f"disruptions-{year}.big.csv",
        )
    )
for year in range(2019, 2026):
    DOWNLOADS.append(
        (
            f"services-{year}",
            f"https://opendata.rijdendetreinen.nl/public/services/services-{year}.csv.gz",
            DATA / f"services-{year}.big.csv.gz",
        )
    )


def download(name: str, url: str, dest: Path) -> None:
    if dest.exists() and dest.stat().st_size > 0:
        print(f"skip {name} ({dest.name}, {dest.stat().st_size / 1024**2:.1f} MiB)")
        return
    print(f"download {name} -> {dest.name}")
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=600) as resp:
        dest.write_bytes(resp.read())
    print(f"  done ({dest.stat().st_size / 1024**2:.1f} MiB)")


for name, url, dest in DOWNLOADS:
    download(name, url, dest)


skip stations (stations-2023-09.big.csv, 0.1 MiB)
skip distances (tariff-distances-2022-01.big.csv, 0.6 MiB)
skip disruptions-2011 (disruptions-2011.big.csv, 0.4 MiB)
skip disruptions-2012 (disruptions-2012.big.csv, 0.5 MiB)
skip disruptions-2013 (disruptions-2013.big.csv, 0.5 MiB)
skip disruptions-2014 (disruptions-2014.big.csv, 0.5 MiB)
skip disruptions-2015 (disruptions-2015.big.csv, 0.6 MiB)
skip disruptions-2016 (disruptions-2016.big.csv, 0.6 MiB)
skip disruptions-2017 (disruptions-2017.big.csv, 1.2 MiB)
skip disruptions-2018 (disruptions-2018.big.csv, 1.5 MiB)
skip disruptions-2019 (disruptions-2019.big.csv, 1.8 MiB)
skip disruptions-2020 (disruptions-2020.big.csv, 1.3 MiB)
skip disruptions-2021 (disruptions-2021.big.csv, 1.4 MiB)
skip disruptions-2022 (disruptions-2022.big.csv, 1.6 MiB)
skip disruptions-2023 (disruptions-2023.big.csv, 1.6 MiB)
skip services-2019 (services-2019.big.csv.gz, 365.1 MiB)
skip services-2020 (services-2020.big.csv.gz, 370.6 MiB)
skip services-2021 (ser

## 2. Stations table (DuckDB)


In [2]:
import duckdb

DB_PATH = "data/homework.big.duckdb"
db = duckdb.connect(DB_PATH)

db.sql("""
CREATE OR REPLACE TABLE stations AS
FROM read_csv('data/stations-2023-09.big.csv', header=true)
""")
db.sql("FROM stations SELECT * LIMIT 3").show()
db.sql("SELECT COUNT(*) FROM stations").show()


┌───────┬─────────┬─────────┬────────────┬──────────────────┬───────────────────────┬──────────────────────┬─────────┬───────────────────────────┬─────────────────┬─────────────────┐
│  id   │  code   │   uic   │ name_short │   name_medium    │       name_long       │         slug         │ country │           type            │     geo_lat     │     geo_lng     │
│ int64 │ varchar │  int64  │  varchar   │     varchar      │        varchar        │       varchar        │ varchar │          varchar          │     double      │     double      │
├───────┼─────────┼─────────┼────────────┼──────────────────┼───────────────────────┼──────────────────────┼─────────┼───────────────────────────┼─────────────────┼─────────────────┤
│   266 │ HT      │ 8400319 │ Den Bosch  │ 's-Hertogenbosch │ 's-Hertogenbosch      │ s-hertogenbosch      │ NL      │ knooppuntIntercitystation │        51.69048 │         5.29362 │
│   269 │ HTO     │ 8400320 │ Dn Bosch O │ 's-Hertogenb. O. │ 's-Hertogenbosch Oost │

## 3. Distances and distances_long (DuckDB tutorial)


In [ ]:
db.sql("""
CREATE OR REPLACE TABLE distances AS
FROM read_csv(
    'data/tariff-distances-2022-01.big.csv',
    header=true,
    nullstr='XXX'
)
""")

db.sql("""
CREATE OR REPLACE TABLE distances_long AS
    UNPIVOT distances
    ON COLUMNS(* EXCLUDE Station)
    INTO NAME other_station VALUE distance
""")

db.sql("""
SELECT Station, other_station, distance
FROM distances_long
LIMIT 5
""").show()


┌─────────┬───────────────┬──────────┐
│ Station │ other_station │ distance │
│ varchar │    varchar    │  int64   │
├─────────┼───────────────┼──────────┤
│ AC      │ AH            │       82 │
│ AC      │ AHP           │       83 │
│ AC      │ AHPR          │       85 │
│ AC      │ AHZ           │       90 │
│ AC      │ AKL           │       71 │
└─────────┴───────────────┴──────────┘



## 4. Disruptions in PostgreSQL


In [4]:
db.sql("INSTALL postgres; LOAD postgres;")

import os

_pg_port = os.environ.get("PGPORT", "5432")
_pg_host = os.environ.get("PGHOST", "localhost")
PG_CONN = f"host={_pg_host} port={_pg_port} user=postgres password=postgres dbname=postgres"

try:
    db.sql("DETACH pg_hw")
except Exception:
    pass

db.sql(f"""
ATTACH '{PG_CONN}' AS pg_hw (TYPE postgres);
""")

db.sql("DROP TABLE IF EXISTS pg_hw.disruptions")

db.sql("""
CREATE TABLE pg_hw.disruptions AS
SELECT *
FROM read_csv('data/disruptions-20*.big.csv', header=true, union_by_name=true)
""")

db.sql("SELECT COUNT(*) FROM pg_hw.disruptions").show()


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        49900 │
└──────────────┘



## 5. Services → single Parquet file and DuckDB table


In [5]:
PARQUET_PATH = "data/services.big.parquet"

db.sql(f"""
COPY (
    SELECT * FROM read_csv('data/services-20*.big.csv.gz', union_by_name=true)
) TO '{PARQUET_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

db.sql(f"""
CREATE OR REPLACE TABLE services AS
SELECT * FROM '{PARQUET_PATH}'
""")

db.sql("SELECT COUNT(*) AS rows FROM services").show()
db.sql('SELECT MIN("Service:Date") AS min_date, MAX("Service:Date") AS max_date FROM services').show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┐
│   rows    │
│   int64   │
├───────────┤
│ 152176324 │
└───────────┘

┌────────────┬────────────┐
│  min_date  │  max_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2019-01-01 │ 2025-12-31 │
└────────────┴────────────┘



## 6. Notes on the data model


Each row is one stop on a route. A full end-to-end service shares `Service:RDT-ID`.
The first stop has `Stop:Arrival time` NULL; the last stop has `Stop:Departure time` NULL.


## 7. Analytics questions


### 7.1 How many trains departed from Amsterdam Centraal overall?


In [6]:
q71 = db.sql("""
SELECT COUNT(*) AS departures_amsterdam_central
FROM services
WHERE "Stop:Station code" = 'ASD'
  AND "Stop:Departure time" IS NOT NULL
  AND COALESCE("Stop:Departure cancelled", false) = false
""")
q71.show()


┌──────────────────────────────┐
│ departures_amsterdam_central │
│            int64             │
├──────────────────────────────┤
│                      1799338 │
└──────────────────────────────┘



**Answer 7.1:** 1,799,338 departures from Amsterdam Centraal (station code ASD, non-cancelled departures).


### 7.2 Average arrival delay by service type (descending)


In [7]:
q72 = db.sql("""
SELECT
    "Service:Type",
    AVG("Stop:Arrival delay") AS avg_arrival_delay
FROM services
WHERE "Stop:Arrival delay" IS NOT NULL
GROUP BY "Service:Type"
ORDER BY avg_arrival_delay DESC
""")
q72.show()


┌──────────────────────┬────────────────────┐
│     Service:Type     │ avg_arrival_delay  │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ Alpen Express        │  31.84254143646409 │
│ Krokus Express       │ 18.304347826086957 │
│ European Sleeper     │ 14.262589044460821 │
│ Nightjet             │   9.42391097403092 │
│ ICE                  │   7.76692176533566 │
│ Eurostar             │  7.302104678961749 │
│ ICE International    │  5.482790573052148 │
│ Thalys               │ 5.3961896909446025 │
│ Int. Trein           │  4.677560708417016 │
│ Nachttrein           │ 3.8370501214396113 │
│     ·                │                 ·  │
│     ·                │                 ·  │
│     ·                │                 ·  │
│ Metro i.p.v. trein   │                0.0 │
│ Tram i.p.v. trein    │                0.0 │
│ stoptrein            │                0.0 │
│ Belbus ipv trein     │                0.0 │
│ Belbus               │          

**Answer 7.2:** Highest average arrival delay by type: Alpen Express (~31.8 min), Krokus Express (~18.3 min), European Sleeper (~14.3 min); Sprinter/Intercity types are typically below ~1.5 min (full table in output above).


### 7.3 Most common disruption cause per year (MODE)


In [8]:
q73 = db.sql("""
SELECT
    year(start_time) AS year,
    mode(statistical_cause_en) AS most_common_cause
FROM pg_hw.disruptions
WHERE statistical_cause_en IS NOT NULL
GROUP BY year(start_time)
ORDER BY year
""")
q73.show()


┌───────┬───────────────────┐
│ year  │ most_common_cause │
│ int64 │      varchar      │
├───────┼───────────────────┤
│  2011 │ broken down train │
│  2012 │ points failure    │
│  2013 │ points failure    │
│  2014 │ broken down train │
│  2015 │ broken down train │
│  2016 │ broken down train │
│  2017 │ broken down train │
│  2018 │ broken down train │
│  2019 │ broken down train │
│  2020 │ broken down train │
│  2021 │ broken down train │
│  2022 │ broken down train │
│  2023 │ broken down train │
└───────┴───────────────────┘
  13 rows         2 columns



**Answer 7.3:** Most frequent statistical cause by year: 2011–2021 and 2014+ mostly *broken down train*; 2012–2013 *points failure*.


### 7.4 Services that started at any Amsterdam station


In [9]:
q74 = db.sql("""
SELECT COUNT(DISTINCT s."Service:RDT-ID") AS services_starting_amsterdam
FROM services s
JOIN stations st ON s."Stop:Station code" = st.code
WHERE s."Stop:Arrival time" IS NULL
  AND st.name_long LIKE 'Amsterdam%'
""")
q74.show()


┌─────────────────────────────┐
│ services_starting_amsterdam │
│            int64            │
├─────────────────────────────┤
│                     1272985 │
└─────────────────────────────┘



**Answer 7.4:** 1,272,985 distinct services started at an Amsterdam station (first stop: `Stop:Arrival time` IS NULL, station name `Amsterdam%`).


### 7.5 Fraction of services with final destination outside the Netherlands


In [10]:
q75 = db.sql("""
WITH last_stops AS (
    SELECT
        "Service:RDT-ID",
        arg_max("Stop:Station code", "Stop:Arrival time") AS end_station_code
    FROM services
    WHERE "Stop:Arrival time" IS NOT NULL
    GROUP BY "Service:RDT-ID"
)
SELECT
    AVG(CASE WHEN st.country != 'NL' THEN 1.0 ELSE 0.0 END) AS fraction_outside_nl
FROM last_stops ls
JOIN stations st ON ls.end_station_code = st.code
""")
q75.show()


┌──────────────────────┐
│ fraction_outside_nl  │
│        double        │
├──────────────────────┤
│ 0.037424226182661724 │
└──────────────────────┘



**Answer 7.5:** About **3.74%** of services end outside the Netherlands (final stop joined to `stations.country`).


### 7.6 Largest distance between NL stations


In [11]:
q76 = db.sql("""
SELECT
    s1.name_long AS station1,
    s2.name_long AS station2,
    dl.distance
FROM distances_long dl
JOIN stations s1 ON dl.Station = s1.code
JOIN stations s2 ON dl.other_station = s2.code
WHERE s1.country = 'NL'
  AND s2.country = 'NL'
  AND dl.distance IS NOT NULL
  AND dl.Station < dl.other_station
ORDER BY dl.distance DESC
LIMIT 5
""")
q76.show()


┌──────────────────┬────────────────────┬──────────┐
│     station1     │      station2      │ distance │
│     varchar      │      varchar       │  int64   │
├──────────────────┼────────────────────┼──────────┤
│ Eemshaven        │ Vlissingen         │      426 │
│ Bad Nieuweschans │ Vlissingen         │      425 │
│ Eemshaven        │ Vlissingen Souburg │      425 │
│ Bad Nieuweschans │ Vlissingen Souburg │      424 │
│ Stavoren         │ Vlissingen         │      421 │
└──────────────────┴────────────────────┴──────────┘



**Answer 7.6:** Largest NL–NL tariff distance: **426** units between **Eemshaven** and **Vlissingen** (425 for Bad Nieuweschans–Vlissingen).


### 7.7 Average arrival delay by operator (bar plot)


In [12]:
import matplotlib.pyplot as plt

df77 = db.sql("""
SELECT
    "Service:Company" AS company,
    AVG("Stop:Arrival delay") AS avg_arrival_delay
FROM services
WHERE "Stop:Arrival delay" IS NOT NULL
  AND "Service:Company" IS NOT NULL
GROUP BY "Service:Company"
ORDER BY avg_arrival_delay DESC
""").df()

ax = df77.plot.bar(
    x="company",
    y="avg_arrival_delay",
    figsize=(10, 5),
    legend=False,
    color="steelblue",
)
ax.set_title("Average arrival delay by train operator")
ax.set_xlabel("Operator")
ax.set_ylabel("Average arrival delay (minutes)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


**Answer 7.7:** Bar chart shows operators sorted by descending average arrival delay (international/night operators at the top, NS regional types lowest).


### 7.8 Disrupted services per year (line plot)


In [13]:
df78 = db.sql("""
WITH stop_times AS (
    SELECT
        "Service:RDT-ID",
        year("Service:Date") AS year,
        "Stop:Station code" AS station_code,
        COALESCE("Stop:Arrival time", "Stop:Departure time") AS stop_time
    FROM services
    WHERE COALESCE("Stop:Arrival time", "Stop:Departure time") IS NOT NULL
)
SELECT year, COUNT(DISTINCT "Service:RDT-ID") AS disrupted_services
FROM stop_times st
WHERE EXISTS (
    SELECT 1
    FROM pg_hw.disruptions d
    WHERE year(d.start_time) = st.year
      AND st.stop_time BETWEEN d.start_time AND d.end_time
      AND list_contains(
          str_split(COALESCE(d.rdt_station_codes, ''), ','),
          st.station_code
      )
)
GROUP BY year
ORDER BY year
""").df()

ax = df78.plot.line(
    x="year",
    y="disrupted_services",
    marker="o",
    figsize=(10, 5),
    title="Number of train services affected by disruptions per year",
)
ax.set_xlabel("Year")
ax.set_ylabel("Distinct services (Service:RDT-ID)")
plt.tight_layout()
plt.show()

df78


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,disrupted_services
0,2019,134387
1,2020,150050
2,2021,165712
3,2022,216768
4,2023,254466


**Answer 7.8:** Disrupted distinct services rose from **134,387 (2019)** to **254,466 (2023)**; see line plot.


### 7.9 Fraction of completely cancelled services per year (line plot)


In [14]:
df79 = db.sql("""
SELECT
    year("Service:Date") AS year,
    AVG(CASE WHEN "Service:Completely cancelled" THEN 1.0 ELSE 0.0 END) AS cancelled_fraction
FROM (
    SELECT DISTINCT
        "Service:RDT-ID",
        "Service:Date",
        "Service:Completely cancelled"
    FROM services
)
GROUP BY year
ORDER BY year
""").df()

ax = df79.plot.line(
    x="year",
    y="cancelled_fraction",
    marker="o",
    figsize=(10, 5),
    title="Fraction of completely cancelled services per year",
)
ax.set_xlabel("Year")
ax.set_ylabel("Fraction cancelled")
plt.tight_layout()
plt.show()

df79


,year,cancelled_fraction
0,2019,0.013212
1,2020,0.086580
2,2021,0.041762
3,2022,0.063572
4,2023,0.058378
5,2024,0.023109
6,2025,0.021683


**Answer 7.9:** Cancellation fraction peaked in **2020 (~8.7%)**, then fell to **~2.2% in 2025**; see line plot.


## 8. Service segment distances


### 8.1–8.2 Build station_connections


In [15]:
db.sql("""
CREATE OR REPLACE TABLE station_connections AS
WITH leg AS (
    SELECT
        "Service:RDT-ID",
        "Stop:Station code" AS start_station_code,
        LEAD("Stop:Station code") OVER (
            PARTITION BY "Service:RDT-ID"
            ORDER BY "Stop:Departure time"
        ) AS end_station_code
    FROM services
    WHERE "Stop:Departure time" IS NOT NULL
),
with_distance AS (
    SELECT
        l."Service:RDT-ID",
        l.start_station_code,
        l.end_station_code,
        dl.distance
    FROM leg l
    JOIN distances_long dl
      ON dl.Station = l.start_station_code
     AND dl.other_station = l.end_station_code
    WHERE l.end_station_code IS NOT NULL
)
SELECT
    "Service:RDT-ID",
    start_station_code,
    end_station_code,
    distance
FROM (
    SELECT *,
        row_number() OVER (
            PARTITION BY start_station_code, end_station_code
            ORDER BY "Service:RDT-ID"
        ) AS rn
    FROM with_distance
) t
WHERE rn = 1
""")

db.sql("SELECT COUNT(*) FROM station_connections").show()
db.sql("FROM station_connections SELECT * LIMIT 5").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         5555 │
└──────────────┘

┌────────────────┬────────────────────┬──────────────────┬──────────┐
│ Service:RDT-ID │ start_station_code │ end_station_code │ distance │
│     int64      │      varchar       │     varchar      │  int64   │
├────────────────┼────────────────────┼──────────────────┼──────────┤
│         738805 │ ASD                │ SHL              │       19 │
│         738858 │ GS                 │ BZL              │        5 │
│         738901 │ ASDM               │ ASSP             │        1 │
│         738889 │ NDB                │ BSMZ             │        1 │
│         738857 │ EHV                │ EHS              │        1 │
└────────────────┴────────────────────┴──────────────────┴──────────┘



### 8.3 Largest distance between a pair of consecutive stations


In [16]:
q83 = db.sql("""
SELECT *
FROM station_connections
ORDER BY distance DESC
LIMIT 5
""")
q83.show()


┌────────────────┬────────────────────┬──────────────────┬──────────┐
│ Service:RDT-ID │ start_station_code │ end_station_code │ distance │
│     int64      │      varchar       │     varchar      │  int64   │
├────────────────┼────────────────────┼──────────────────┼──────────┤
│        7122512 │ VSS                │ HNP              │      411 │
│        7122512 │ WK                 │ VSS              │      407 │
│        7122512 │ MDB                │ WK               │      402 │
│        7122706 │ VSS                │ SK               │      391 │
│        7122512 │ SK                 │ MDB              │      386 │
└────────────────┴────────────────────┴──────────────────┴──────────┘



**Answer 8.3:** Largest consecutive segment in `station_connections`: **411** tariff units (e.g. VSS→HNP on service 7122512).


### 8.4 Histogram of inter-station distances


In [17]:
dist_df = db.sql("SELECT distance FROM station_connections WHERE distance IS NOT NULL").df()

ax = dist_df["distance"].plot.hist(bins=50, figsize=(10, 5), color="steelblue", edgecolor="white")
ax.set_title("Histogram of distances between consecutive stations on train routes")
ax.set_xlabel("Distance (tariff units)")
ax.set_ylabel("Number of station pairs")
plt.tight_layout()
plt.show()


**Answer 8.4:** Histogram of deduplicated consecutive-station distances (see plot).
